<h1 style="text-align:center">Airborne Object Tracking Dataset</h1>

# 🤫 Setting up

In [ ]:
import json
import random
import os, sys
from IPython.display import display, clear_output, HTML
from random import randrange, choice
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=25,25
import numpy as np
import seaborn as sns

# Because Life, Universe and Everything!
random.seed(42)

def mdprint(text):
    display({
        'text/markdown': text,
        'text/plain': text
    }, raw=True)

!git clone http://gitlab.aicrowd.com/amazon-prime-air/airborne-detection-starter-kit.git
os.chdir("airborne-detection-starter-kit/data")

Cloning into 'airborne-detection-starter-kit'...
remote: Enumerating objects: 337, done.
remote: Total 337 (delta 0), reused 0 (delta 0), pack-reused 337
Receiving objects: 100% (337/337), 21.90 MiB | 9.04 MiB/s, done.
Resolving deltas: 100% (139/139), done.


In [ ]:
# Dataset for Airborne Object Tracking Dataset
sys.path.append(os.path.dirname(os.path.realpath(os.getcwd())))
sys.path.append(os.path.dirname(os.path.realpath(os.getcwd())) + "/core")
!pip install -r ../requirements.txt > /dev/null
from core.dataset import Dataset
notebook_path = os.path.dirname(os.path.realpath("__file__"))

local_path = notebook_path + '/part2'
s3_path = 's3://airborne-obj-detection-challenge-training/part2/'
dataset = Dataset(local_path, s3_path)

ERROR: botocore 1.20.95 has requirement urllib3<1.27,>=1.25.4, but you'll have urllib3 1.24.3 which is incompatible.


2021-06-16 19:29:18.507 | INFO     | core.dataset:load_gt:20 - Loading ground truth...
2021-06-16 19:29:18.508 | INFO     | core.file_handler:download_file_if_needed:33 - [download_from_s3] File not found locally, downloading: ImageSets/groundtruth.json


In [ ]:
# Download validation flights
# - e0d815053c1c46cfbd0b586b72718feb
# - ac23cb93c5c242d2b1bf0633fae9b1e6

flight = dataset.get_flight_by_id("e0d815053c1c46cfbd0b586b72718feb")
flight.download()

flight = dataset.get_flight_by_id("ac23cb93c5c242d2b1bf0633fae9b1e6")
flight.download()

# ⏱ Generate the predictions

In [ ]:
os.chdir("/content/airborne-detection-starter-kit/")
!ln -s $PWD/data/part2/Images $PWD/data/val

In [ ]:
!python test.py

Successfully generated predictions!


# 👀 Time to evaluate locally!

In [ ]:
# Cleanup...
!rm -rf data/evaluation/
!mkdir -p data/evaluation/gt
!mkdir -p data/evaluation/result

In [ ]:
# Generate groundtruth.json for relevant flights
def generate_partial_gt():
    flights = ['e0d815053c1c46cfbd0b586b72718feb', 'ac23cb93c5c242d2b1bf0633fae9b1e6']
    gt = json.loads(open("data/part2/ImageSets/groundtruth.json").read())
    for sample in list(gt['samples'].keys()):
        if sample not in flights:
            del gt['samples'][sample]

    with open(("data/evaluation/gt/groundtruth.json"), 'w') as fp:
        json.dump(gt, fp)

In [ ]:
# Transfer generated results to metrics codebase bbox format
def convert_and_copy_generated_results_to_metrics_folder():
    flight_results = json.loads(open("data/results/run0/result.json").read())
    for i in range(len(flight_results)):
        for j in range(len(flight_results[i]['detections'])):
            x = flight_results[i]['detections'][j]['x']
            y = flight_results[i]['detections'][j]['y']
            w = flight_results[i]['detections'][j]['w'] - x
            h = flight_results[i]['detections'][j]['h'] - y


            flight_results[i]['detections'][j]['x'] = x + w/2
            flight_results[i]['detections'][j]['y'] = y + h/2
            flight_results[i]['detections'][j]['w'] = w
            flight_results[i]['detections'][j]['h'] = h

    with open("data/evaluation/result/result.json", 'w') as fp:
        json.dump(flight_results, fp)

In [ ]:
# Let's run the metrics codebase!!
generate_partial_gt()
convert_and_copy_generated_results_to_metrics_folder()
!python core/metrics/run_airborne_metrics.py --dataset-folder data/evaluation/gt --results-folder data/evaluation/result --summaries-folder data/evaluation/summaries

2021-06-16 19:39:16,872:INFO:run_airborne_metrics.py:160 Encounter ground truth: data/evaluation/gt/groundtruth_with_encounters_maxRange700_maxGap3_minEncLen30.csv
2021-06-16 19:39:16,873:INFO:calculate_encounters.py:83 Asserting data/evaluation/gt/groundtruth.json format
2021-06-16 19:39:16,873:INFO:pandas_utils.py:87 Reading ground truth
2021-06-16 19:39:16,873:INFO:pandas_utils.py:61 Reading provided data/evaluation/gt/groundtruth.json
2021-06-16 19:39:16,873:INFO:pandas_utils.py:68 Loading .json
2021-06-16 19:39:16,881:INFO:pandas_utils.py:75 Normalizing json. This operation is time consuming. The result .csv will be saved Please consider providing .csv file next time
2021-06-16 19:39:16,958:INFO:calculate_encounters.py:268 Saving groundtruth in .csv format, please use .csv in the future
2021-06-16 19:39:16,981:INFO:calculate_encounters.py:273 Filtering ground truth to get intruders in the specified range <= 700m.
2021-06-16 19:39:17,005:INFO:utils.py:157 NumExpr defaulting to 4 th

In [ ]:
!cat data/evaluation/summaries/*_for_ranking.csv

#,Algorithm,Score,FPPI,AFDR
0,result_metrics_min_track_len_0,0.7001715688383829,0.1137974155898291,0.0
#,Algorithm,Score,HFAR,"EDR All, Tracking"
0,result_metrics_min_track_len_0,0.7001715688383829,30.0,0.0


## Try out SiamMot results OR your own generated result files

You can obviously run things locally, but in case you want to try out on Colab, you can download like in example below

In [ ]:
# Cleanup...
!rm -rf data/evaluation/
!mkdir -p data/evaluation/gt
!mkdir -p data/evaluation/result

In [ ]:
# Downloading SiamMot result files for validation flights
!wget https://gitlab.aicrowd.com/snippets/37555/raw -O data/results/run0/result.json

--2021-06-16 19:53:03--  https://gitlab.aicrowd.com/snippets/37555/raw
Resolving gitlab.aicrowd.com (gitlab.aicrowd.com)... 18.194.109.98, 18.193.11.37
Connecting to gitlab.aicrowd.com (gitlab.aicrowd.com)|18.194.109.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84561 (83K) [text/plain]
Saving to: ‘data/results/run0/result.json’

data/results/run0/r 100%[===================>]  82.58K  --.-KB/s    in 0.1s    

2021-06-16 19:53:04 (593 KB/s) - ‘data/results/run0/result.json’ saved [84561/84561]



In [ ]:
# Let's run the metrics codebase!!
generate_partial_gt()
convert_and_copy_generated_results_to_metrics_folder()
!python core/metrics/run_airborne_metrics.py --dataset-folder data/evaluation/gt --results-folder data/evaluation/result --summaries-folder data/evaluation/summaries

2021-06-16 19:53:15,009:INFO:run_airborne_metrics.py:160 Encounter ground truth: data/evaluation/gt/groundtruth_with_encounters_maxRange700_maxGap3_minEncLen30.csv
2021-06-16 19:53:15,010:INFO:calculate_encounters.py:83 Asserting data/evaluation/gt/groundtruth.json format
2021-06-16 19:53:15,010:INFO:pandas_utils.py:87 Reading ground truth
2021-06-16 19:53:15,010:INFO:pandas_utils.py:61 Reading provided data/evaluation/gt/groundtruth.json
2021-06-16 19:53:15,010:INFO:pandas_utils.py:68 Loading .json
2021-06-16 19:53:15,019:INFO:pandas_utils.py:75 Normalizing json. This operation is time consuming. The result .csv will be saved Please consider providing .csv file next time
2021-06-16 19:53:15,089:INFO:calculate_encounters.py:268 Saving groundtruth in .csv format, please use .csv in the future
2021-06-16 19:53:15,117:INFO:calculate_encounters.py:273 Filtering ground truth to get intruders in the specified range <= 700m.
2021-06-16 19:53:15,131:INFO:utils.py:157 NumExpr defaulting to 4 th

In [ ]:
!cat data/evaluation/summaries/*_for_ranking.csv

#,Algorithm,Score,FPPI,AFDR
0,result_metrics_min_track_len_0,0.9851007461547852,0.0,0.9852941176470589
#,Algorithm,Score,HFAR,"EDR All, Tracking"
0,result_metrics_min_track_len_0,0.9851007461547852,0.0,1.0


## The tricky bits

- Use groundtruth.json with flights information only for the flights you are evaluating on. (example shared in this notebook)
- Metrics codebase use different bbox format v/s the one in groundtruth & submissions. Use the `convert_and_copy_generated_results_to_metrics_folder()` function to convert, example above.


## Getting different results in your submission v/s local run for SiamMot?

- Check the pytorch version, maskrcnn, etc. We have provided Dockerfile in the repository which generate exactly the same results.json as downloaded in this colab
- Have you downloaded full flight or partial flight in your local?
- Check if you are using MIN_TRACK_LEN = 30, MIN_SCORE = 0.985

Are you sure everything is same?

Please let us know on Discord or Discourse, we can help in debugging further, cheers!

👋 👋 👋 👋